In [1]:
%cd /p/project/deepacf/maelstrom/langguth1/downscaling_benchmark/postprocess/jupyter_notebooks

/p/project/deepacf/maelstrom/langguth1/downscaling_benchmark/postprocess/jupyter_notebooks


In [3]:
%matplotlib inline

import os, sys
base_dir = "../../"
sys.path.extend([f"{base_dir}/models", f"{base_dir}/utils", f"{base_dir}/handle_data", f"{base_dir}/postprocess"])
import numpy as np
import xarray as xr
import pandas as pd
from statistical_evaluation import Scores
from statistical_evaluation import perform_block_bootstrap_metric as bootstrapping
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt

In [12]:
# Basic settings
results_basedir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results" #f"{base_dir}results"
data_dir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset" 
exps_eval = ["sha_unet_benchmark_t2m", "sha_wgan_benchmark_t2m", "deepru_benchmark_t2m", "ankit_swinir", "bilinear"]
ref_exp = "bilinear"
nboots = 1000
metric = "rmse"

plt_fname = "rmse_meta_postprocessing.png"

In [13]:
# initialize dicts
rmse_tall_exps = {}
skill_tall_exps = {}

In [14]:
for i, exp in enumerate(exps_eval):
    print(f"Run evaluation for {exp}...")
    if exp == "bilinear":
        ds = xr.open_dataset(os.path.join(data_dir, "downscaling_benchmark_t2m_test.nc"))
        score_engine = Scores(ds["t2m_in"], ds["t_2m_tar"], ["rlat", "rlon"])
    else:
        ds = xr.open_dataset(os.path.join(results_basedir, exp, "postprocessed_ds_test.nc"))
        score_engine = Scores(ds["t_2m_tar_fcst"], ds["t_2m_tar_ref"], ["rlat", "rlon"])
    
    rmse_tall_exps[exp] = score_engine(metric)

Run evaluation for sha_unet_benchmark_t2m...
Run evaluation for sha_wgan_benchmark_t2m...
Run evaluation for deepru_benchmark_t2m...
Run evaluation for ankit_swinir...
Run evaluation for bilinear...


In [15]:
# get reference
rmse_ref = rmse_tall_exps[ref_exp]

# initialize DataArrays to store results
exps_eval.remove(ref_exp)
nexps = len(exps_eval)

skill_avg = xr.DataArray(np.zeros(nexps), coords={"experiment": exps_eval}, dims="experiment")
skill_boot = xr.DataArray(np.zeros(nexps*nboots).reshape(nexps, nboots),
                          coords={"experiment": exps_eval, "iboot": np.arange(nboots)},
                          dims=["experiment", "iboot"])

for i, exp in enumerate(exps_eval):
    skill_tall_exps[exp] = (rmse_ref - rmse_tall_exps[exp])/rmse_ref
    skill_avg[i], skill_boot[i,:] = skill_tall_exps[exp].mean(), bootstrapping(skill_tall_exps[exp], "time", 120)

100%|██████████| 1000/1000 [00:01<00:00, 928.10it/s]


In [89]:
skill_boot[0,:]

<xarray.DataArray (iboot: 1000)>
array([0.48085539, 0.47788803, 0.4872935 , 0.47550395, 0.47403537,
       0.4845044 , 0.46989454, 0.47262303, 0.48814883, 0.47604924,
       0.47883803, 0.48014768, 0.47862605, 0.48596769, 0.47595601,
       0.47771634, 0.46443223, 0.4850704 , 0.46721276, 0.48035485,
       0.46617112, 0.47598952, 0.48140484, 0.47357479, 0.47621511,
       0.47182698, 0.45900563, 0.47557005, 0.47117691, 0.46818317,
       0.48049193, 0.48417971, 0.47807934, 0.47531682, 0.48279064,
       0.48170007, 0.46427365, 0.48006723, 0.48628834, 0.47833241,
       0.47179337, 0.47300373, 0.47041049, 0.48557208, 0.47733481,
       0.48631107, 0.48490401, 0.47341551, 0.48220333, 0.48677715,
       0.4822829 , 0.4815491 , 0.48461809, 0.47316323, 0.47925975,
       0.47599396, 0.47036323, 0.47744889, 0.4712998 , 0.48526793,
       0.46972245, 0.47294843, 0.47485134, 0.47979497, 0.47135282,
       0.47641762, 0.47685059, 0.48260596, 0.47856353, 0.47774201,
       0.48047873, 0.46675808, 0.48415303, 0.48408697, 0.49380812,
       0.48150261, 0.48443172, 0.47791003, 0.47361911, 0.46140568,
       0.4820156 , 0.47576989, 0.48391003, 0.48457131, 0.47705981,
       0.4835402 , 0.47687281, 0.49098627, 0.47757936, 0.48633367,
       0.48259102, 0.48927479, 0.46977418, 0.48366936, 0.48209896,
       0.46781433, 0.4727759 , 0.4740592 , 0.48266978, 0.47084359,
...
       0.47873728, 0.4755086 , 0.47358349, 0.47499744, 0.47136015,
       0.48641204, 0.48716231, 0.47180134, 0.48745648, 0.47175672,
       0.47885799, 0.48150428, 0.47711539, 0.47057475, 0.4759879 ,
       0.48457091, 0.47821794, 0.4844308 , 0.48066708, 0.48456483,
       0.47729827, 0.47237412, 0.47496661, 0.48249755, 0.47980423,
       0.47593396, 0.47009656, 0.47591135, 0.48019404, 0.47136784,
       0.47568389, 0.47284331, 0.47795492, 0.48366396, 0.47396878,
       0.48164965, 0.49022145, 0.4719818 , 0.47266883, 0.47122109,
       0.48102306, 0.48495461, 0.47953929, 0.47532888, 0.47873644,
       0.47799398, 0.47995644, 0.47208806, 0.47878325, 0.48010935,
       0.46577271, 0.48448647, 0.4741582 , 0.4735438 , 0.47763085,
       0.47669453, 0.4824436 , 0.4720118 , 0.49099288, 0.47748332,
       0.48078932, 0.48323896, 0.47409486, 0.47620159, 0.482788  ,
       0.47825139, 0.47472188, 0.48031942, 0.47894725, 0.47098165,
       0.48329643, 0.48469025, 0.48196679, 0.48228862, 0.47654262,
       0.4730101 , 0.48759136, 0.47499156, 0.48027114, 0.48293775,
       0.47444722, 0.47111895, 0.47564019, 0.47963019, 0.46957415,
       0.48322648, 0.48316738, 0.47963034, 0.47677914, 0.47679826,
       0.48105405, 0.46933316, 0.48796257, 0.47708337, 0.47656181,
       0.46938668, 0.47962789, 0.48284417, 0.48151839, 0.47843267])
Coordinates:
    experiment  <U24 'sha_unet_benchmark_t2m_2'
  * iboot       (iboot) int64 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999

In [16]:
def plot_skills(skill_avg, skill_boot, plt_fname, labels=["U-Net (Sha)", "WGAN (Sha)", "DeepRU", "SwinIR"],
                metric="RMSE"):
    fs = 16

    # create figure
    fig, ax = plt.subplots(1, 1)
    # create box-plot
    bp = ax.boxplot(skill_boot.T, labels=labels, patch_artist=True)
    # configure plot
    ax.set_ylim(.4, .5)

    ax.set_title("")
    ax.set_ylabel(f"Skill {metric}", fontsize=fs)
    ax.tick_params(axis="both", which="both", direction="out", labelsize=fs-2)

    colors = ['pink', 'lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2.)
        
    plt.rcParams['text.usetex'] = True
    ax.text(0.7, 0.05, r"$\overline{RMSE}_{ref}$="+f"{rmse_ref.mean():.2f}K",
        transform=ax.transAxes,
        color='k', fontsize=fs-2)

    fig.savefig(plt_fname, bbox_inches="tight")
    plt.tight_layout()
    fig.savefig(plt_fname)
    plt.close(fig)

In [18]:
plot_skills(skill_avg, skill_boot, plt_fname, labels=["Sha U-Net", "Sha WGAN", "DeepRU", "SwinIR"])